# Supervised fine-tuning: LoRA rank versus full tuning
This notebook trains Qwen2.5-0.5B four times on the same 1,000 UltraChat conversations: LoRA ranks 1, 4, and 16, then full-weight tuning. Use one fresh NVIDIA L4 runtime for the entire experiment. The requirements deliberately retain Colab's CUDA-matched PyTorch installation.

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path
REPO = 'Dense-and-Mixture-of-Experts-vLLM'
REPO_URL = 'https://github.com/itisaby/Dense-and-Mixture-of-Experts-vLLM.git'
if Path('/content', REPO).exists():
    !git -C /content/{REPO} pull --ff-only
else:
    !git clone {REPO_URL} /content/{REPO}
os.chdir(Path('/content') / REPO)
print('Working directory:', Path.cwd())

In [ ]:
%pip install -q -r requirements_sft.txt
import torch, transformers, datasets, peft
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'transformers:', transformers.__version__, 'datasets:', datasets.__version__, 'peft:', peft.__version__)

## Run the matched experiment
The command writes each completed variant immediately. The full run includes dataset download, four training jobs, periodic evaluation, and held-out generation.

In [ ]:
!python scripts/sft_experiment.py --variants all --output-dir results/sft --overwrite

In [ ]:
!python scripts/plot_sft_results.py --input-dir results/sft

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
display(pd.read_csv('results/sft/metrics.csv'))
display(Image('figures/sft_loss_curves.png'))
display(Image('figures/sft_resource_comparison.png'))
display(Markdown(Path('results/sft/qualitative_generations.md').read_text()))

In [ ]:
from google.colab import files
!zip -r sft_artifacts.zip results/sft figures/sft_loss_curves.* figures/sft_resource_comparison.*
files.download('sft_artifacts.zip')